# Watching the readout form: $W_U$ parameter-trajectory crosscoders


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hematteo/learning-to-read-out/blob/main/notebooks/02_wu_trajectory_crosscoders.ipynb)

The companion notebook ([`01_availability_expression_lag.ipynb`](01_availability_expression_lag.ipynb))
showed that hidden states carry task signal *before* the readout — the unembedding matrix $W_U$ —
expresses it. This notebook is about the instrument the paper builds to watch the readout itself
get organized.

**The idea.** Each pretraining checkpoint $t$ gives a snapshot $W_U^{(t)} \in \mathbb{R}^{V \times d}$.
Row $v$ is the direction the model dots against the hidden state to score token $v$. Instead of
fitting a separate sparse autoencoder per checkpoint (which gives $K$ unrelated dictionaries),
we fit **one crosscoder across the whole trajectory**: a single sparse code $a(v) \in \mathbb{R}^{d_{sae}}$
per vocabulary row, with a *per-snapshot decoder* $W_D^{(t)}$:

$$W_U^{(t)}[v] \;\approx\; \sum_j a_j(v)\, W_D^{(t)}[j] \qquad \text{for all } t \text{ simultaneously.}$$

Features are identified across time *by construction*, so the per-snapshot decoder norm
$\lVert W_D^{(t)}[j] \rVert$ is a well-defined **lifecycle trajectory** for feature $j$: when it
appears, grows, reorganizes, persists.

| token | meaning |
|---|---|
| $K$ | number of snapshots (paper: 32; here: 8) |
| $V$ | vocabulary rows = the crosscoder's batch axis (Pythia: 50,304) |
| $d$ | model hidden dim (Pythia-160M: 768) |
| $d_{sae}$ | dictionary size = expansion_factor × $d$ |

**What this notebook does:**

- **Part A** — trains a crosscoder on a *toy* $W_U$ trajectory with planted structure and checks
  it recovers both the planted dictionary and the planted growth schedule (~1 min, CPU).
- **Part B** — trains a small but *real* instrument: Pythia-160M $W_U$ snapshots across 8
  pretraining checkpoints (~3 GB of downloads on first run).
- **Part C** — uses it to do the paper's core analyses in miniature: feature lifecycles,
  formation timing, and the vocabulary families features encode.
- **Part D** *(optional)* — loads the paper's full released instruments.

**Runtime:** Part A ~1 min anywhere. Parts B–C: ~10–12 min on a Colab T4 GPU
(`Runtime → Change runtime type → T4`) or Apple Silicon; on pure CPU the notebook
automatically drops to a rougher 30-epoch instrument (~20 min). The trained instrument is
cached to disk, so re-runs skip training entirely. All seeded.

In [ ]:
# Setup: locate (or clone) the repo and install the vendored SAE library.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hematteo/learning-to-read-out"


def find_repo_root():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "src" / "crosscoder" / "wu_adapter.py").exists():
            return cand
    return None


ROOT = find_repo_root()
if ROOT is None:  # fresh Colab runtime -> clone
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    ROOT = Path.cwd() / "learning-to-read-out"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# The vendored llamascopium library decorates unused model classes with
# @torch.autocast(device_type="cuda") at import time; harmless on CPU.
import logging
import warnings

warnings.filterwarnings("ignore", message="CUDA is not available")
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

try:
    import llamascopium  # noqa: F401  vendored OpenMOSS Language-Model-SAEs (see docs/THIRD_PARTY.md)
except ImportError:  # first Colab run: ~2-4 min (pulls transformer-lens & friends)
    %pip install -q {ROOT}/lib/Language-Model-SAEs

import matplotlib.pyplot as plt
import torch

from src.core.data import get_device
from src.core.repro import seed_everything
from src.crosscoder.wu_adapter import (
    batch_iter,
    build_crosscoder,
    load_snapshots,
    preprocess_snapshots,
    quick_quality,
    train,
)

seed_everything(0)
DEVICE = get_device()  # cuda > mps > cpu
print(f"repo:   {ROOT}")
print(f"torch {torch.__version__} | device: {DEVICE}")

## Part A · The method, on a toy with known answers

Real $W_U$ snapshots have unknown structure — a bad place to learn what an instrument does. So
we *plant* the structure: 16 random directions ("atoms"), 256 fake vocabulary rows that are
sparse mixes of **exactly 3 atoms each** (loadings bounded away from zero, so every planted
coefficient is recoverable in principle), and a smooth per-checkpoint growth schedule
(scale 0.8 → 1.2 across 8 snapshots) standing in for readout drift during pretraining. Then we
check the crosscoder recovers **both** the atoms and the growth schedule.

(Same shape of setup as the repo's CPU demo,
[`examples/minimal_crosscoder.py`](../examples/minimal_crosscoder.py); the hyperparameters
follow the paper's W_U recipe, `configs/runs/pythia-160m_wu_d24576_seed0.yaml`.)

In [ ]:
# Tiny synthetic problem. Real runs use K=32 snapshots, V~50k vocab rows, d=768..4096.
N_SNAPSHOTS = 8  # K: number of W_U snapshots stacked on the hook axis
VOCAB = 256  # V: vocabulary rows, the crosscoder's "batch" axis
D_MODEL = 64  # d: model hidden dim (W_U row width)
N_ATOMS = 16  # rows are sparse mixes of this many shared directions
K_ACTIVE = 3  # planted atoms per row
EXPANSION_FACTOR = 4.0  # d_sae = expansion_factor * d_model = 256 features

seed_everything(0)
atoms = torch.randn(N_ATOMS, D_MODEL)  # (n_atoms, d_model) planted dictionary
support = torch.zeros(VOCAB, N_ATOMS)  # (V, n_atoms) exactly K_ACTIVE atoms per row
for v in range(VOCAB):
    support[v, torch.randperm(N_ATOMS)[:K_ACTIVE]] = 1.0
signs = torch.randint(0, 2, (VOCAB, N_ATOMS)).float() * 2 - 1
coeffs = support * signs * (0.5 + torch.rand(VOCAB, N_ATOMS))  # |loading| in [0.5, 1.5]
drift = torch.linspace(0.8, 1.2, N_SNAPSHOTS)  # planted per-checkpoint growth
toy = drift.view(-1, 1, 1) * (coeffs @ atoms).unsqueeze(0)  # (K, V, d)
toy = toy + 0.05 * torch.randn_like(toy)  # small observation noise

# Per-snapshot row-mean centering, as in all real runs (stats are invertible).
toy, _ = preprocess_snapshots(toy, mode="center")
print("toy snapshot stack:", tuple(toy.shape), "(K, V, d)")

In [ ]:
# Untrained EV as the reference point, then train (~30 s on CPU).
untrained = build_crosscoder(N_SNAPSHOTS, D_MODEL, expansion_factor=EXPANSION_FACTOR, device="cpu")
ev0 = quick_quality(untrained, toy, batch_size=64, device="cpu")["explained_variance"]

cc_toy = train(
    toy,
    expansion_factor=EXPANSION_FACTOR,
    l1_coefficient=0.3,  # paper W_U recipe (configs/runs/pythia-160m_wu_d24576_seed0.yaml)
    tanh_stretch_coefficient=1.0,
    n_epochs=1200,  # V=256 -> 4 steps/epoch; sparsification needs the steps
    batch_size=64,
    device="cpu",
    seed=0,
    log_every=1200,
)
q = quick_quality(cc_toy, toy, batch_size=64, device="cpu")
print(
    f"\nEV {ev0:.3f} (untrained) -> {q['explained_variance']:.3f} (trained)   "
    f"L0 = {q['mean_l0']:.1f} active features/row (planted: {K_ACTIVE} atoms/row)   "
    f"d_sae = {q['d_sae']}"
)

High explained variance at an L0 near the planted 3 atoms/row means the crosscoder found *a*
sparse description. Did it find *the* one we planted? Compare every planted atom against the
learned decoder directions at the final snapshot:

In [ ]:
with torch.no_grad():
    W_D_toy = cc_toy.W_D.detach().float().cpu()  # (K, d_sae, d_model) per-snapshot decoder
toy_norms = W_D_toy.norm(dim=-1)  # (K, d_sae) feature size per snapshot

A = atoms / atoms.norm(dim=-1, keepdim=True)  # (n_atoms, d)
D = W_D_toy[-1] / W_D_toy[-1].norm(dim=-1, keepdim=True).clamp_min(1e-8)  # (d_sae, d)
cos = (A @ D.T).abs()  # (n_atoms, d_sae)
best_cos, best_feat = cos.max(dim=1)  # best-matching learned feature per atom

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.2), constrained_layout=True)
axes[0].bar(range(N_ATOMS), best_cos.numpy(), color="C0")
axes[0].axhline(1.0, ls=":", c="gray", lw=1)
axes[0].set_xlabel("planted atom")
axes[0].set_ylabel("best |cosine| to a learned feature")
axes[0].set_ylim(0, 1.05)
im = axes[1].imshow(cos[:, best_feat].numpy(), vmin=0, vmax=1, cmap="viridis")
axes[1].set_xlabel("matched learned feature")
axes[1].set_ylabel("planted atom")
fig.colorbar(im, ax=axes[1], shrink=0.85, label="|cosine|")
plt.show()

print(f"mean best-match |cos| = {best_cos.mean():.3f}, worst atom = {best_cos.min():.3f}  (1.0 = perfect recovery)")

A bright diagonal in the matched block = each planted atom has a dedicated learned feature.

Now the part that makes it a *trajectory* instrument. We planted a growth schedule
(every direction scaled 0.8 → 1.2 across snapshots). The crosscoder never saw it explicitly —
but since the sparse code $a(v)$ is shared across snapshots, all per-checkpoint change must be
absorbed by the per-snapshot decoders. Each matched feature's decoder-norm trajectory should
therefore reproduce the planted schedule:

In [ ]:
traj = toy_norms[:, best_feat]  # (K, n_atoms) matched-feature size across snapshots
traj_rel = traj / traj.mean(dim=0, keepdim=True)
drift_rel = (drift / drift.mean()).numpy()

fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.plot(traj_rel.numpy(), color="C0", alpha=0.35)
ax.plot(drift_rel, "k--", lw=2, label="planted growth schedule")
ax.set_xlabel("snapshot index")
ax.set_ylabel("decoder norm (relative)")
ax.set_title("Recovered feature lifecycles vs. planted drift")
ax.legend()
plt.show()

Blue lines (one per recovered atom) hug the dashed planted schedule: **per-snapshot decoder
norms read out each feature's lifecycle.** That single fact powers everything in Part C.

## Part B · A real instrument: Pythia-160M across pretraining

Now the real thing, scaled to a notebook. We take 8 of the paper's 32 canonical checkpoint steps
(`DEFAULT_STEPS_32`), pull each Pythia-160M checkpoint's unembedding $W_U$ — the repo's
`load_snapshots` downloads each checkpoint (~375 MB) once, caches the extracted $W_U$ (~150 MB)
under `local_snapshots/`, and never re-downloads — and train a crosscoder over the
$(8, 50304, 768)$ stack.

Honest framing: the paper's instruments use all 32 snapshots, $d_{sae}{=}24{,}576$, 300 epochs,
multiple seeds, and post-hoc validation (held-out checkpoints, per-snapshot-SAE and dense-PCA
baselines, an EV-vs-L0 Pareto sweep behind the operating point); the settings of record reach
EV ≈ 0.92 at L0 ≈ 286 (`configs/runs/pythia-160m_wu_d24576_seed0.yaml`). Our mini-instrument
(8 snapshots, $d_{sae}{=}3072$, 100 epochs) lands around **EV ≈ 0.56 at L0 ≈ 206** — a coarser
microscope, but its features are already interpretable, which is what Part C needs.

In [ ]:
MODEL = "EleutherAI/pythia-160m"
REAL_STEPS = [0, 256, 1000, 2000, 8000, 21000, 75000, 143000]  # 8 of the canonical 32
EXPANSION = 4.0  # d_sae = 3072 (paper: 32.0 -> 24576)
# ~4,900 steps at 100 epochs; sparsification needs them. On pure CPU we drop to a
# rougher 30-epoch instrument by default — raise it back if you have the patience.
EPOCHS = 100 if DEVICE != "cpu" else 30
print(f"training budget: {EPOCHS} epochs on {DEVICE}")
CACHE = ROOT / "local_snapshots" / "pythia-160m"

snaps_raw, steps = load_snapshots(MODEL, steps=REAL_STEPS, cache_dir=CACHE)
print("W_U stack:", tuple(snaps_raw.shape), "(K, V, d)")

# center_scale: per-snapshot centering + E[||x||^2] = d scaling, as in the paper recipe.
# (Plain centering leaves raw Pythia W_U rows tiny next to the JumpReLU threshold and
# the dictionary collapses to all-dead — try it.)
snaps, _stats = preprocess_snapshots(snaps_raw, mode="center_scale")
del snaps_raw

In [ ]:
# Train the mini instrument (~10 min on a T4 or Apple Silicon at 100 epochs).
# The result is cached to disk: re-running this cell, or the whole notebook,
# loads it back in seconds. Set RETRAIN = True to force a fresh run.
RETRAIN = False
CKPT = ROOT / "local_snapshots" / f"wu_cc_demo_160m_d{int(EXPANSION * snaps.shape[-1])}_ep{EPOCHS}.pt"

seed_everything(0)
if CKPT.exists() and not RETRAIN:
    cc = build_crosscoder(len(steps), snaps.shape[-1], expansion_factor=EXPANSION, device=DEVICE)
    cc.load_state_dict(torch.load(CKPT, map_location="cpu", weights_only=True))
    print(f"loaded cached demo instrument {CKPT.name} (set RETRAIN = True to retrain)")
else:
    cc = train(
        snaps,
        expansion_factor=EXPANSION,
        l1_coefficient=0.3,  # paper W_U recipe: lambda=0.3, tanh_stretch=1.0
        tanh_stretch_coefficient=1.0,
        lr=1e-4,
        n_epochs=EPOCHS,
        batch_size=1024,
        device=DEVICE,
        seed=0,
        log_every=500,
    )
    torch.save(cc.state_dict(), CKPT)

q = quick_quality(cc, snaps, batch_size=2048, device=DEVICE)
print(
    f"\nEV = {q['explained_variance']:.3f}   L0 = {q['mean_l0']:.1f} active features/row   "
    f"dead = {q['dead_rate']:.0%} of d_sae = {q['d_sae']}"
)

## Part C · Doing science with it

### C.1 Feature lifecycles and formation timing

Exactly as in the toy: the per-snapshot decoder norm $\lVert W_D^{(t)}[j] \rVert$ is feature
$j$'s lifecycle. We normalize each living feature by its final size and ask **when it reached
half of it** — a crude "formation step".

In [ ]:
K = len(steps)
with torch.no_grad():
    norms = cc.W_D.detach().float().norm(dim=-1).cpu()  # (K, d_sae) lifecycle trajectories

final = norms[-1]
alive = final > 0.05 * final.max()  # ignore dead / never-grown features
rel_all = norms / final.clamp_min(1e-8)  # (K, d_sae) normalized to final size
formed_all = (rel_all >= 0.5).int().argmax(dim=0)  # first snapshot at >= half final size
print(f"alive features: {int(alive.sum())}/{norms.shape[1]}")

step_labels = [f"{s//1000}k" if s >= 1000 else str(s) for s in steps]
cmap = plt.get_cmap("viridis")
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.4), constrained_layout=True)

alive_idx = torch.nonzero(alive).squeeze(1)
sample = alive_idx[torch.randperm(len(alive_idx))[:60]]
for j in sample.tolist():
    axes[0].plot(range(K), rel_all[:, j].numpy(), color=cmap(formed_all[j].item() / max(K - 1, 1)), alpha=0.45)
axes[0].axhline(0.5, ls=":", c="gray", lw=1)
axes[0].set_xticks(range(K), step_labels)
axes[0].set_xlabel("pretraining step")
axes[0].set_ylabel("decoder norm / final")
axes[0].set_title("60 random feature lifecycles (color = formation step)")

counts = torch.bincount(formed_all[alive], minlength=K)
axes[1].bar(range(K), counts.numpy(), color="C0")
axes[1].set_xticks(range(K), step_labels)
axes[1].set_xlabel("formation step (first snapshot at ≥ half final size)")
axes[1].set_ylabel("# features")
axes[1].set_title("When do readout features form?")
plt.show()

Nearly all surviving features reach half their final size within the first few thousand
steps — a single-digit percentage of the 143k-step run. The paper's 32-snapshot grid resolves
this window finely, localizes the take-off around step ~1k, and sharpens it into a *causal*
claim with activation patching around the localization window
(`experiments/causal/temporal_localization_patching/`); the lifecycle shapes
(formation / reorganization / persistence, the "wishbone") are classified in
`experiments/lifecycle/feature_lifecycle_trajectories/`.

### C.2 What do the features mean? Vocabulary families

A feature's sparse code $a_j(v)$ lives over vocabulary rows, so "what does feature $j$ do" has a
directly readable answer: **which tokens activate it**. Encode all 50k rows once, then look at
each feature's top tokens — grouped by the formation step from C.1, because the families
themselves have lifecycles. (In our runs, early-formers are punctuation/quote/bracket families,
while some whitespace and byte-fragment families keep forming late into training.)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)


@torch.no_grad()
def encode_rows(crosscoder, snapshots, batch_size=4096):
    """Shared per-row feature activations a(v). Returns (V, d_sae) on CPU."""
    crosscoder.eval()
    chunks = []
    for batch in batch_iter(snapshots, crosscoder.cfg.hook_points, batch_size, shuffle=False, device=DEVICE):
        x, enc_kw, _ = crosscoder.prepare_input(batch)
        acts = crosscoder.encode(x, **enc_kw)  # (B, K, d_sae) -- identical across K by construction
        chunks.append(acts[:, 0, :].float().cpu())
    return torch.cat(chunks)


acts = encode_rows(cc, snaps)  # (V, d_sae)
print("row activations:", tuple(acts.shape), "(V, d_sae)")


def top_tokens(j, k=10):
    vals, idx = acts[:, j].topk(k)
    return [(tokenizer.decode([t]), round(v, 2)) for t, v in zip(idx.tolist(), vals.tolist())]


# Gallery, grouped by WHEN each feature formed: families have lifecycles.
strength = acts.max(dim=0).values
order = strength.argsort(descending=True)
groups = {
    "formed early (by step 2k)": formed_all <= steps.index(2000),
    "formed mid (by step 8k)": formed_all == steps.index(8000),
    "formed late (after step 8k)": formed_all > steps.index(8000),
}
for label, mask in groups.items():
    idx = torch.nonzero(mask & alive).squeeze(1)
    print(f"\n{label}: {len(idx)} features")
    for j in idx[strength[idx].argsort(descending=True)[:4]].tolist():
        toks = ", ".join(repr(t) for t, _ in top_tokens(j, k=8))
        print(f"  feature {j:>4} | formed step {steps[int(formed_all[j])]:>6} | {toks}")

In [ ]:
FEATURE = int(order[0])  # <-- edit me: any index in [0, d_sae)

vals, idx = acts[:, FEATURE].topk(15)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), constrained_layout=True)
axes[0].barh(range(15), vals.numpy()[::-1], color="C0")
axes[0].set_yticks(range(15), [repr(tokenizer.decode([t])) for t in idx.tolist()][::-1], fontsize=8)
axes[0].set_xlabel("activation $a_j(v)$")
axes[0].set_title(f"feature {FEATURE}: top tokens")
axes[1].plot(range(K), norms[:, FEATURE].numpy(), "o-", color="C1")
axes[1].set_xticks(range(K), step_labels)
axes[1].set_xlabel("pretraining step")
axes[1].set_ylabel("decoder norm")
axes[1].set_title(f"feature {FEATURE}: lifecycle")
plt.show()

Together, the two panels are the paper's core move in one picture: a feature is a coherent
**vocabulary family** (left) with a **lifecycle in pretraining time** (right). At paper scale
this becomes §5.4's family-emergence analysis; whether such features are *load-bearing* for
behavior is then tested causally by editing them in the readout
(`experiments/causal/sparse_feature_causal_tests/`,
`experiments/causal/contrastive_task_feature_rescue/`).

## Part D · The paper's released instruments *(optional)*

The crosscoders behind the thesis figures (160M/1B/6.9B, $K{=}32$, plus per-snapshot-SAE
baselines and the λ-sweep grid) are released as `safetensors` + config sidecars. Set
`RELEASE_REPO` once they are published and this cell loads the real 160M instrument and redraws
the formation histogram at paper scale — no snapshot downloads needed, since lifecycles live
entirely in $W_D$.

In [ ]:
RELEASE_REPO = None  # TODO(release): published HF repo id, e.g. "<org>/parameter-trajectory-crosscoders"
RELEASE_REPO_TYPE = "model"  # TODO(release): "model" or "dataset", matching how the artifacts are hosted

if RELEASE_REPO is None:
    print("Set RELEASE_REPO to load the paper's released crosscoders (see docs/DATA.md).")
else:
    from huggingface_hub import hf_hub_download

    from src.core.model_specs import DEFAULT_STEPS_32
    from src.crosscoder.checkpoints import load_checkpoint

    sub = "pythia-160m/W_U/cross-snapshot-32/d24576"
    path = hf_hub_download(RELEASE_REPO, f"{sub}/seed0.safetensors", repo_type=RELEASE_REPO_TYPE)
    hf_hub_download(RELEASE_REPO, f"{sub}/seed0.config.json", repo_type=RELEASE_REPO_TYPE)
    ck = load_checkpoint(path)

    W_D_paper = ck.state_dict["W_D"].float()  # (32, 24576, 768)
    paper_steps = ck.steps or DEFAULT_STEPS_32
    norms_p = W_D_paper.norm(dim=-1)  # (32, 24576)
    final_p = norms_p[-1]
    alive_p = final_p > 0.05 * final_p.max()
    formed_p = ((norms_p / final_p.clamp_min(1e-8)) >= 0.5).int().argmax(dim=0)

    fig, ax = plt.subplots(figsize=(8, 3.2))
    counts_p = torch.bincount(formed_p[alive_p], minlength=len(paper_steps))
    ax.bar(range(len(paper_steps)), counts_p.numpy(), color="C0")
    ax.set_xticks(range(len(paper_steps)), [str(s) for s in paper_steps], rotation=90, fontsize=7)
    ax.set_xlabel("formation step")
    ax.set_ylabel("# features")
    ax.set_title(f"Paper-scale instrument: {tuple(W_D_paper.shape)} | {int(alive_p.sum())} alive features")
    plt.show()

## Where each part lives in the repository

| Notebook section | Repository code | Paper claim it backs |
|---|---|---|
| Part A (toy) | [`examples/minimal_crosscoder.py`](../examples/minimal_crosscoder.py), `src/crosscoder/wu_adapter.py` | the instrument itself |
| Part B (training) | `scripts/train/train_crosscoder.py`, `experiments/crosscoders/crosscoder_main/` | §5.1 multi-model validation |
| Instrument quality / capacity choice | `experiments/capacity/pareto_frontier_ev_l0/` | EV-vs-L0 operating point |
| Baselines it beats | `experiments/baselines/` (per-snapshot SAEs, dense PCA) | sparse + cross-snapshot is necessary |
| C.1 lifecycles | `experiments/lifecycle/feature_lifecycle_trajectories/` | §5.2 sparse lifecycle |
| C.1 formation timing | `experiments/causal/temporal_localization_patching/` | §5.3 localization near step 1k |
| C.2 vocabulary families | `experiments/crosscoders/crosscoder_main/` | §5.4 family emergence |
| Causal load-bearing tests | `experiments/causal/sparse_feature_causal_tests/`, `.../contrastive_task_feature_rescue/` | features matter for behavior |

[`docs/REPRODUCE.md`](../docs/REPRODUCE.md) maps every thesis figure to the script that produces
its metrics; [`experiments.yaml`](../experiments.yaml) is the machine-readable manifest;
[`docs/DATA.md`](../docs/DATA.md) documents external assets and the storage layout.

**Citation:** see [`CITATION.cff`](../CITATION.cff) (thesis Chapter 4; paper link added on release).